# ShoreSense - Module 3: Guidance Q&A (RAG)

## Install

## Imports

In [3]:
import os
import glob
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

## Load API Key

In [4]:
load_dotenv()
print("Key found:", os.getenv("GROQ_API_KEY") is not None)

Key found: True


## Load PDFs

In [8]:
docs = []

for file in glob.glob(r"data\protocols/*.pdf"):
    pages = PyPDFLoader(file).load()
    for page in pages:
        page.metadata["source"] = os.path.basename(file)
    docs.extend(pages)

print(len(docs), "pages loaded")

29 pages loaded


## Compare Two Chunk Sizes

In [9]:
for size in [500, 1000]:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    print(size, "->", len(splitter.split_documents(docs)), "chunks")

500 -> 179 chunks
1000 -> 90 chunks


## Split Documents

In [10]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(docs)
print(len(chunks))
print(chunks[0].metadata)

179
{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-14T10:45:13+00:00', 'author': 'marine-triage project', 'keywords': '', 'moddate': '2026-10-14T10:45:13+00:00', 'subject': '(unspecified)', 'title': 'Species Fact Sheet: Common Bottlenose Dolphin (Tursiops truncatus) - Conservation and Management', 'trapped': '/False', 'source': 'bottlenose_dolphin__conservation_management__noaa.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}


## Embeddings

In [11]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Build and Save FAISS Index

In [12]:
db = FAISS.from_documents(chunks, embeddings)
db.save_local("faiss_index")

## Load FAISS Index

In [13]:
db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)

## Test Retrieval

In [14]:
results = db.similarity_search("How close can I go to a stranded dolphin?", k=3)

for r in results:
    print(r.metadata["source"])
    print(r.page_content[:200])
    print("---")

general__first_response_guide__synthetic.pdf
Synthetic educational guide compiled from NOAA and WTI documents in this corpus. Not official rescue guidance.
Page 3
5. Distances by animal and region
 Animal
Minimum distance
Region / file
North Atl
---
right_whale__how_you_can_help__noaa.pdf
7. Note on use in the triage project
 - Species-specific distance rule: 500 yards for right whales. Other NOAA guidance in this project gives 50
yards for bottlenose dolphins, 100 yards for sperm whal
---
southeast_protected_marine_life__mariner_guide__noaa.pdf
chase or encircle. Never position a vessel between
mother and calf.
Harbors, bays, sounds and
estuaries, nearshore
waters and the open ocean
Hotline for injured, entangled or stranded dolphins and wha
---


## Load LLM (Groq - GPT OSS)

In [15]:
llm = ChatGroq(model="openai/gpt-oss-20b", temperature=0)

## Prompt

In [16]:
prompt = ChatPromptTemplate.from_template("""You are ShoreSense, an educational assistant for marine animal stranding guidance.

Rules:
1. Answer ONLY from the context below.
2. Cite the file name of every source, like [file_name.pdf].
3. If documents from different regions give different guidance, state the difference by region.
4. If the answer is not in the context, reply exactly: The documents do not cover this. Please contact your local wildlife authorities.
5. These are synthetic educational documents, not official rescue guidance.

Context:
{context}

Question: {question}

Answer:""")

## Answer Question Function

In [17]:
def answer_question(question, species=None):
    query = question
    if species:
        query = question + " " + species

    docs = db.similarity_search(query, k=4)
    context = "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)

    chain = prompt | llm
    response = chain.invoke({"context": context, "question": query})
    return response.content, docs

## Test In-Scope Questions

In [ ]:
from IPython.display import display, Markdown

In [22]:
questions = [
    "What should I do if I find an entangled whale?",
    "How far should I stay from a stranded dolphin?",
    "Who should I call to report an entangled marine mammal?",
]

for q in questions:
    answer, sources = answer_question(q)
    print("Q:", q)
    print("A:")
    display(Markdown(answer))
    print("Sources:", set(s.metadata["source"] for s in sources))
    print("=" * 60)

Q: What should I do if I find an entangled whale?
A:


If you spot an entangled whale, you should:

1. **Report it immediately** – give the U.S. Coast Guard the details on VHF channel 16 or use the 24‑hour regional hotline for your area.  
2. **Provide as much information as possible** – date, time, exact location, species (if known), the animal’s condition and behavior, and any gear or debris you see. Take photos or video from a safe, legal distance.  
3. **Do not approach or try to free the whale yourself** – this can be dangerous for both you and the animal.  
4. **Stay nearby only if it is safe** – remaining at a safe distance can help responders, but leave if you feel unsafe.

**Regional hotlines**

| Region | Phone number |
|--------|--------------|
| Maine through Virginia | (866) 755‑6622 |
| North Carolina through Texas, and the Caribbean | (877) WHALE‑HELP / (877) 942‑5343 |
| California, Oregon and Washington | (877) WHALE‑HELP / (877) 942‑5343 |

*(Numbers are from the NOAA guidance; the synthetic guide confirms the same procedure.)*  

Sources: [entangled_marine_mammal__how_to_report__noaa.pdf], [general__first_response_guide__synthetic.pdf]

Sources: {'entangled_marine_mammal__how_to_report__noaa.pdf', 'general__first_response_guide__synthetic.pdf'}
Q: How far should I stay from a stranded dolphin?
A:


You should keep a minimum distance of **50 yards** from a stranded dolphin (and other dolphins or porpoises). This is the recommended minimum distance in both the general first‑response guide and the Southeast protected‑marine‑life mariner guide.  

- **general__first_response_guide__synthetic.pdf** – “Bottlenose dolphin (and other dolphins and porpoises) 50 yards; limit viewing to 30 minutes or less.”  
- **southeast_protected_marine_life__mariner_guide__noaa.pdf** – “Bottlenose dolphin: Stay at least 50 yards away. Do not approach.”

Sources: {'bottlenose_dolphin__conservation_management__noaa.pdf', 'monk_seal__species_fact_sheet__noaa.pdf', 'southeast_protected_marine_life__mariner_guide__noaa.pdf', 'general__first_response_guide__synthetic.pdf'}
Q: Who should I call to report an entangled marine mammal?
A:


Call the U.S. Coast Guard on VHF channel 16 or the 24‑hour regional hotline for your area.  
- **Maine through Virginia:** (866) 755‑6622  
- **North Carolina through Texas, and the Caribbean:** (877) WHALE‑HELP / (877) 942‑5343  
- **California, Oregon and Washington:** use the regional hotline (phone number not listed in the documents)  

These hotlines are listed in *entangled_marine_mammal__how_to_report__noaa.pdf* and *general__first_response_guide__synthetic.pdf*.

Sources: {'entangled_marine_mammal__how_to_report__noaa.pdf', 'general__first_response_guide__synthetic.pdf'}


## Test Out-of-Scope Questions

In [19]:
out_questions = [
    "What is the best recipe for grilled fish?",
    "How do I repair a boat engine?",
]

for q in out_questions:
    answer, sources = answer_question(q)
    print("Q:", q)
    print("A:", answer)
    print("=" * 60)

Q: What is the best recipe for grilled fish?
A: The documents do not cover this. Please contact your local wildlife authorities.
Q: How do I repair a boat engine?
A: The documents do not cover this. Please contact your local wildlife authorities.
